# Global Temperature Regression Analysis

**Objective:** Build a regression-based analytical pipeline to identify the most influential drivers of global temperature change.

This notebook collects data from scientific sources, preprocesses it, trains regression models, and uses feature importance to determine key environmental factors.

**Data Sources:**
- NOAA Global Monitoring Laboratory (CO₂, CH₄, N₂O)
- NASA GISS (Temperature anomalies, Solar irradiance)
- Our World in Data (Anthropogenic factors)

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import requests
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Create directories
data_dir = Path('../data')
reports_dir = Path('../reports')
data_dir.mkdir(exist_ok=True)
reports_dir.mkdir(exist_ok=True)

## 2. Data Collection from Scientific Sources

We'll download data from three main sources:
- NOAA: Greenhouse gases (CO₂, CH₄, N₂O)
- NASA GISS: Temperature anomalies and solar irradiance
- OWID: Anthropogenic factors like CO₂ emissions

In [ ]:
# Data URLs
data_urls = {
    'co2_monthly': 'https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.txt',
    'ch4_monthly': 'https://gml.noaa.gov/webdata/ccgg/trends/ch4/ch4_mm_gl.txt',
    'n2o_monthly': 'https://gml.noaa.gov/webdata/ccgg/trends/n2o/n2o_mm_gl.txt',
    'temperature_anomaly': 'https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.txt',
    'solar_irradiance': 'https://www.ncei.noaa.gov/data/total-solar-irradiance/access/SOLAR_IRRADIANCE_V02R.txt',
    'co2_emissions': 'https://ourworldindata.org/grapher/co-emissions-by-sector.csv',
    'energy_consumption': 'https://ourworldindata.org/grapher/energy-consumption-by-source-and-region.csv'
}

def download_file(url, filename):
    """Download a file from URL"""
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        filepath = data_dir / filename
        with open(filepath, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded {filename}")
        return True
    except Exception as e:
        print(f"Error downloading {filename}: {e}")
        return False

# Download all files
for name, url in data_urls.items():
    filename = f"{name}.txt" if not url.endswith('.csv') else f"{name}.csv"
    download_file(url, filename)

print("Data collection completed!")

## 3. Data Preprocessing and Cleaning

Now we'll load and clean each dataset, standardizing formats and handling missing values.

In [ ]:
def load_noaa_gas_data(filename, gas_name):
    """Load NOAA greenhouse gas data"""
    filepath = data_dir / filename
    if not filepath.exists():
        print(f"File {filename} not found")
        return pd.DataFrame()

    df = pd.read_csv(filepath, delim_whitespace=True, comment='#', header=None,
                     names=['year', 'month', 'decimal_date', 'average', 'trend', 'days'])
    df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
    df = df[['date', 'average']].rename(columns={'average': gas_name})
    df = df.set_index('date')
    return df

def load_nasa_temperature():
    """Load NASA GISS temperature anomaly data"""
    filepath = data_dir / 'temperature_anomaly.txt'
    if not filepath.exists():
        print("Temperature file not found")
        return pd.DataFrame()

    df = pd.read_csv(filepath, delim_whitespace=True, skiprows=1, header=None)
    columns = ['year'] + [f'month_{i}' for i in range(1,13)]
    df.columns = columns

    df_melted = df.melt(id_vars=['year'], var_name='month', value_name='anomaly')
    df_melted['month'] = df_melted['month'].str.replace('month_', '').astype(int)
    df_melted['date'] = pd.to_datetime(df_melted[['year', 'month']].assign(day=1))
    df_melted = df_melted[['date', 'anomaly']].set_index('date')
    df_melted['anomaly'] = pd.to_numeric(df_melted['anomaly'], errors='coerce')
    return df_melted

def load_solar_irradiance():
    """Load solar irradiance data"""
    filepath = data_dir / 'solar_irradiance.txt'
    if not filepath.exists():
        print("Solar irradiance file not found")
        return pd.DataFrame()

    df = pd.read_csv(filepath, delim_whitespace=True, comment=';')
    df['date'] = pd.to_datetime(df[['year', 'month', 'day']])
    df = df[['date', 'irradiance']].set_index('date')
    return df

def load_owid_data(filename, value_col):
    """Load Our World in Data CSV"""
    filepath = data_dir / filename
    if not filepath.exists():
        print(f"File {filename} not found")
        return pd.DataFrame()

    df = pd.read_csv(filepath)
    df_world = df[df['Entity'] == 'World'].copy()
    df_world['date'] = pd.to_datetime(df_world['Year'], format='%Y')
    df_world = df_world[['date', value_col]].set_index('date')
    return df_world

# Load all datasets
print("Loading datasets...")
co2_df = load_noaa_gas_data('co2_monthly.txt', 'co2_ppm')
ch4_df = load_noaa_gas_data('ch4_monthly.txt', 'ch4_ppb')
n2o_df = load_noaa_gas_data('n2o_monthly.txt', 'n2o_ppb')
temp_df = load_nasa_temperature()
solar_df = load_solar_irradiance()
co2_emissions_df = load_owid_data('co2_emissions.csv', 'Annual CO₂ emissions')
energy_df = load_owid_data('energy_consumption.csv', 'Primary energy consumption (TWh)')

print(f"CO2 data: {len(co2_df)} rows")
print(f"Temperature data: {len(temp_df)} rows")
print(f"CO2 emissions data: {len(co2_emissions_df)} rows")

## 4. Feature Engineering

Create additional features that might help explain temperature variation.

In [ ]:
# Merge all datasets
dfs = [co2_df, ch4_df, n2o_df, temp_df, solar_df, co2_emissions_df, energy_df]
merged_df = pd.concat(dfs, axis=1, join='outer')

# Fill missing values
gas_cols = ['co2_ppm', 'ch4_ppb', 'n2o_ppb']
merged_df[gas_cols] = merged_df[gas_cols].fillna(method='ffill')
merged_df['anomaly'] = merged_df['anomaly'].interpolate()

# Remove rows with too many missing values
merged_df = merged_df.dropna(thresh=len(merged_df.columns) * 0.5)

# Feature engineering
merged_df['time_since_baseline'] = (merged_df.index - merged_df.index.min()).days / 365.25

# Growth rates (annual)
for col in gas_cols:
    merged_df[f'{col}_growth_rate'] = merged_df[col].pct_change(12)

# Moving averages
merged_df['co2_ma_12'] = merged_df['co2_ppm'].rolling(12).mean()
merged_df['temp_ma_12'] = merged_df['anomaly'].rolling(12).mean()

print(f"Merged dataset shape: {merged_df.shape}")
print(f"Date range: {merged_df.index.min()} to {merged_df.index.max()}")
print(f"Columns: {list(merged_df.columns)}")

## 5. Exploratory Data Analysis

Let's visualize the data to understand trends and relationships.

In [ ]:
# Time series plot
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(merged_df.index, merged_df['anomaly'], color='red', linewidth=2)
axes[0].set_title('Global Temperature Anomaly (°C)')
axes[0].set_ylabel('Anomaly (°C)')
axes[0].grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(merged_df.index, merged_df['co2_ppm'], label='CO₂', color='blue')
ax2.set_ylabel('CO₂ (ppm)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

ax2_twin = ax2.twinx()
ax2_twin.plot(merged_df.index, merged_df['ch4_ppb'], label='CH₄', color='green', linestyle='--')
ax2_twin.set_ylabel('CH₄ (ppb)', color='green')
ax2_twin.tick_params(axis='y', labelcolor='green')

ax3 = axes[2]
if 'irradiance' in merged_df.columns:
    ax3.plot(merged_df.index, merged_df['irradiance'], label='Solar Irradiance', color='orange')
ax3.set_title('Solar Irradiance')
ax3.set_xlabel('Year')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(reports_dir / 'time_series_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Correlation matrix
key_vars = ['anomaly', 'co2_ppm', 'ch4_ppb', 'n2o_ppb']
if 'irradiance' in merged_df.columns:
    key_vars.append('irradiance')
corr_df = merged_df[key_vars].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_df, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig(reports_dir / 'correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Train/Test Split

Split the data for model training and evaluation.

In [ ]:
# Prepare features and target
df_model = merged_df.dropna(subset=['anomaly']).copy()
feature_cols = [col for col in df_model.columns if col != 'anomaly']
X = df_model[feature_cols]
y = df_model['anomaly']

# Fill missing features
X = X.fillna(X.mean())

# Train/test split (70/30, maintaining time order)
split_idx = int(len(X) * 0.7)
X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Features: {len(feature_cols)}")

## 7. Linear Regression Model Training

In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
y_pred_train_lr = lr_model.predict(X_train)
y_pred_test_lr = lr_model.predict(X_test)

# Metrics
lr_results = {
    'MSE_Train': mean_squared_error(y_train, y_pred_train_lr),
    'MSE_Test': mean_squared_error(y_test, y_pred_test_lr),
    'R2_Train': r2_score(y_train, y_pred_train_lr),
    'R2_Test': r2_score(y_test, y_pred_test_lr)
}

print("Linear Regression Results:")
for metric, value in lr_results.items():
    print(f"{metric}: {value:.4f}")

# Coefficients
coefficients = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 10 Features by Absolute Coefficient:")
print(coefficients.head(10))

## 8. Random Forest and XGBoost Training

In [ ]:
# Train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Train XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

# Predictions
y_pred_test_rf = rf_model.predict(X_test)
y_pred_test_xgb = xgb_model.predict(X_test)

# Feature importances
rf_importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

xgb_importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Random Forest Top 10 Features:")
print(rf_importances.head(10))

print("\nXGBoost Top 10 Features:")
print(xgb_importances.head(10))

## 9. Model Evaluation and Feature Importance

In [ ]:
# Model comparison
models_results = {
    'Linear Regression': lr_results,
    'Random Forest': {
        'MSE_Test': mean_squared_error(y_test, y_pred_test_rf),
        'R2_Test': r2_score(y_test, y_pred_test_rf)
    },
    'XGBoost': {
        'MSE_Test': mean_squared_error(y_test, y_pred_test_xgb),
        'R2_Test': r2_score(y_test, y_pred_test_xgb)
    }
}

results_df = pd.DataFrame(models_results).T
print("Model Performance Comparison:")
print(results_df)

# Plot feature importance for Random Forest
plt.figure(figsize=(10, 6))
plt.bar(range(10), rf_importances['Importance'].head(10))
plt.xticks(range(10), rf_importances['Feature'].head(10), rotation=45, ha='right')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.savefig(reports_dir / 'rf_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot coefficients for Linear Regression
plt.figure(figsize=(10, 6))
top_coeffs = coefficients.head(10)
plt.bar(range(10), top_coeffs['Coefficient'])
plt.xticks(range(10), top_coeffs['Feature'], rotation=45, ha='right')
plt.title('Linear Regression Standardized Coefficients')
plt.tight_layout()
plt.savefig(reports_dir / 'lr_coefficients.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Example Rows After Preprocessing

Here are three example rows from the final processed dataset:

In [ ]:
# Display example rows
print("Example rows from the processed dataset:")
display(df_model.sample(3))

# Save final dataset
df_model.to_csv(data_dir / 'final_processed_data.csv')
print(f"\nFinal dataset saved with {len(df_model)} rows and {len(df_model.columns)} columns")

print("\nAnalysis complete! Check the reports/ directory for visualizations.")